## Task-adaptive pretraining (TAPT)

### Colab Setup

In [2]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

Mounted at /content/drive


### Key Imports

In [3]:
import torch

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.loader_twd_labelled import load_splits
from models.dapt import dapt
from models.plm_finetune import finetune
from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
SEEDS = SHAH_SEEDS
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Continued pretraining (one adapted model per seed)

In [4]:
# TAPT pretrains on the task's own training sentences, labels ignored. 100
# epochs follows Gururangan et al. 2020 section 4.1. one model per seed, since
# each seed has a different train split -- unlike DAPT, which is seed-independent.
ARM = "tapt:roberta-large"
ENC = "roberta-large"
EPOCHS = 100

for seed in SEEDS:
    save_dir = str(RESULTS_DIR / "models" / f"tapt-s{seed}")
    if os.path.isdir(save_dir):
        print(f"{ARM} seed {seed}: already adapted, skipping")
        continue
    train, _ = load_splits("benchmark", seed=seed)
    sentences = train["sentence"].to_list()
    print(f"{ARM} seed {seed}: {len(sentences):,} sentences, {EPOCHS} epochs", flush=True)
    dapt(
        sentences,
        model_name=SHAH_PLM[ENC]["model_name"],
        epochs=EPOCHS,
        save_dir=save_dir,
        device=DEVICE,
        verbose=True,
    )

Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt:roberta-large seed 5768: 1,984 sentences, 100 epochs


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

    saved -> /content/drive/MyDrive/thesis/models/tapt-s78516
Seed 944601 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
tapt:roberta-large seed 944601: 1,984 sentences, 100 epochs


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

    tokenizing 1,984 sentences...
    training: 62 steps/epoch x 100 epoch(s)
    epoch 0: mlm loss 1.3300
    epoch 1: mlm loss 1.2010
    epoch 2: mlm loss 1.2302
    epoch 3: mlm loss 1.1902
    epoch 4: mlm loss 1.1250
    epoch 5: mlm loss 1.1068
    epoch 6: mlm loss 1.0612
    epoch 7: mlm loss 0.9908
    epoch 8: mlm loss 1.0181
    epoch 9: mlm loss 1.0272
    epoch 10: mlm loss 0.9580
    epoch 11: mlm loss 0.9563
    epoch 12: mlm loss 0.9317
    epoch 13: mlm loss 0.8977
    epoch 14: mlm loss 0.9007
    epoch 15: mlm loss 0.8938
    epoch 16: mlm loss 0.9581
    epoch 17: mlm loss 0.9359
    epoch 18: mlm loss 0.9068
    epoch 19: mlm loss 0.8517
    epoch 20: mlm loss 0.8371
    epoch 21: mlm loss 0.8107
    epoch 22: mlm loss 0.7985
    epoch 23: mlm loss 0.8248
    epoch 24: mlm loss 0.7767
    epoch 25: mlm loss 0.7799
    epoch 26: mlm loss 0.7742
    epoch 27: mlm loss 0.7329
    epoch 28: mlm loss 0.7117
    epoch 29: mlm loss 0.6958
    epoch 30: mlm loss 0.7243
  

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    saved -> /content/drive/MyDrive/thesis/models/tapt-s944601


### Fine-tune adapted encoders (3 seeds)

In [5]:
cfg = SHAH_PLM[ENC]

for seed in SEEDS:
    if already_done(OUT, force=FORCE, model=ARM, corpus="twd", seed=seed):
        print(f"{ARM} seed {seed}: already done, skipping")
        continue
    train, test = load_splits("benchmark", seed=seed)
    model, tok_, metrics = finetune(
        train,
        model_name=str(RESULTS_DIR / "models" / f"tapt-s{seed}"),
        lr=cfg["lr"],
        batch_size=cfg["batch_size"],
        seed=seed,
        test_df=test,
        device=DEVICE,
        verbose=True,
    )
    save_result(
        OUT,
        model=ARM,
        corpus="twd",
        seed=seed,
        epochs=metrics["epochs"],
        weighted_f1=round(metrics["test_f1"], 4),
        macro_f1=round(metrics["test_macro_f1"], 4),
    )
    print(f"{ARM} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
    del model, tok_
    torch.cuda.empty_cache()

Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/thesis/models/tapt-s5768
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=1.0074  acc=0.4874  wF1=0.4606  mF1=0.4204  es=0  21.1s
    epoch  1: val CE=0.7703  acc=0.7121  wF1=0.7139  mF1=0.7026  es=0  20.7s
    epoch  2: val CE=0.9364  acc=0.6136  wF1=0.6145  mF1=0.6148  es=1  19.6s
    epoch  3: val CE=0.7084  acc=0.7475  wF1=0.7490  mF1=0.7444  es=0  19.8s
    epoch  4: val CE=0.9903  acc=0.7222  wF1=0.7222  mF1=0.7062  es=1  19.8s
    epoch  5: val CE=0.9802  acc=0.7222  wF1=0.7254  mF1=0.7148  es=2  19.9s
    epoch  6: val CE=0.9628  acc=0.7273  wF1=0.7304  mF1=0.7172  es=3  19.7s
    epoch  7: val CE=1.2988  acc=0.7197  wF1=0.7160  mF1=0.6959  es=4  19.8s
    epoch  8: val CE=1.2961  acc=0.7197  wF1=0.7195  mF1=0.7050  es=5  19.9s
    epoch  9: val CE=1.2674  acc=0.7197  wF1=0.7201  mF1=0.7062  es=6  19.7s
    epoch 10: val CE=1.4532  acc=0.7323  wF1=0.7297  mF1=0.7152  es=7  19.9s
tapt:roberta-large seed 5768: macro=0.6912
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


...

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/thesis/models/tapt-s78516
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=0.8475  acc=0.5758  wF1=0.5894  mF1=0.5612  es=0  19.3s
    epoch  1: val CE=0.7197  acc=0.6894  wF1=0.6962  mF1=0.6709  es=0  19.8s
    epoch  2: val CE=0.8442  acc=0.7096  wF1=0.7140  mF1=0.6868  es=0  19.6s
    epoch  3: val CE=0.8923  acc=0.6843  wF1=0.6917  mF1=0.6701  es=1  19.1s
    epoch  4: val CE=0.9548  acc=0.6894  wF1=0.6934  mF1=0.6703  es=2  19.0s
    epoch  5: val CE=1.1888  acc=0.7323  wF1=0.7302  mF1=0.7025  es=0  19.5s
    epoch  6: val CE=1.3963  acc=0.6970  wF1=0.6996  mF1=0.6757  es=1  19.2s
    epoch  7: val CE=1.4548  acc=0.7222  wF1=0.7228  mF1=0.6939  es=2  19.0s
    epoch  8: val CE=1.4736  acc=0.7146  wF1=0.7194  mF1=0.6941  es=3  19.0s
    epoch  9: val CE=1.5337  acc=0.7096  wF1=0.7156  mF1=0.6886  es=4  18.8s
    epoch 10: val CE=1.2883  acc=0.7197  wF1=0.7266  mF1=0.7007  es=5  19.1s
    epoch 11: val CE=1.2924  acc=0.7348  wF1=0.7328  mF1=0.7034  es=0  19.4s
    epoch 12: val CE=1.4056  acc=0.7298  wF1=0.7309  mF1=0.7011  es=1  19.2s

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/thesis/models/tapt-s944601
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=0.8669  acc=0.6035  wF1=0.6064  mF1=0.5964  es=0  20.4s
    epoch  1: val CE=0.7324  acc=0.6818  wF1=0.6869  mF1=0.6776  es=0  20.4s
    epoch  2: val CE=0.8282  acc=0.7020  wF1=0.7034  mF1=0.6891  es=0  19.9s
    epoch  3: val CE=0.9867  acc=0.6995  wF1=0.7046  mF1=0.6948  es=0  19.6s
    epoch  4: val CE=1.2379  acc=0.7020  wF1=0.7027  mF1=0.6882  es=1  19.5s
    epoch  5: val CE=1.3409  acc=0.6970  wF1=0.7016  mF1=0.6899  es=2  19.5s
    epoch  6: val CE=1.3505  acc=0.7298  wF1=0.7293  mF1=0.7126  es=0  19.8s
    epoch  7: val CE=1.3114  acc=0.7222  wF1=0.7244  mF1=0.7134  es=0  19.5s
    epoch  8: val CE=1.4763  acc=0.7298  wF1=0.7325  mF1=0.7202  es=0  19.8s
    epoch  9: val CE=1.3489  acc=0.7222  wF1=0.7247  mF1=0.7135  es=1  19.3s
    epoch 10: val CE=1.4823  acc=0.7323  wF1=0.7313  mF1=0.7149  es=2  19.4s
    epoch 11: val CE=1.2540  acc=0.7121  wF1=0.7169  mF1=0.7067  es=3  19.5s
    epoch 12: val CE=1.6205  acc=0.6970  wF1=0.6925  mF1=0.6761  es=4  19.4s